Tester

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_rows = 100

df_clean = pd.DataFrame({
    "feature_a": np.random.normal(loc=0.0, scale=1.0, size=n_rows),
    "feature_b": np.random.uniform(low=0, high=10, size=n_rows),
    "feature_c": np.random.poisson(lam=5, size=n_rows),
    "feature_d": np.random.normal(loc=10.0, scale=2.0, size=n_rows),
    "feature_e": np.random.binomial(n=1, p=0.3, size=n_rows),
})

df_clean.head()

,feature_a,feature_b,feature_c,feature_d,feature_e
0,0.496714,4.174110,6,6.994470,0
1,-0.138264,2.221078,2,9.478530,1
2,0.647689,1.198654,4,7.749162,0
3,1.523030,3.376152,9,9.701927,0
4,-0.234153,9.429097,7,9.986693,0


In [2]:
n_error_rows = 20

# Sample rows from the clean dataframe
df_error = df_clean.sample(n_error_rows, random_state=1).reset_index(drop=True)

# --- Inject errors ---

# Additive constant error
df_error["feature_a"] = df_error["feature_a"] + 2.5

# Scaling error
df_error["feature_b"] = df_error["feature_b"] * 1.8

# Increased noise
df_error["feature_c"] = df_error["feature_c"] + np.random.normal(
    loc=0, scale=2.0, size=n_error_rows
)

# Missing values (randomly drop ~30%)
mask_missing = np.random.rand(n_error_rows) < 0.3
df_error.loc[mask_missing, "feature_d"] = np.nan

# Logical / data-entry error: flip binary with noise
flip_mask = np.random.rand(n_error_rows) < 0.2
df_error.loc[flip_mask, "feature_e"] = 1 - df_error.loc[flip_mask, "feature_e"]

df_error.head()


,feature_a,feature_b,feature_c,feature_d,feature_e
0,2.280328,9.877208,1.254739,NaN,0
1,1.691506,12.819226,7.587316,9.061705,0
2,1.442289,9.643944,3.399279,11.055530,0
3,2.857113,12.454114,1.390426,NaN,0
4,2.172338,6.618884,4.987768,5.260557,0


In [3]:
# cleaner (uses inference)
import pandas as pd

from conformal_data_cleaning.cleaner.autogluon import ConformalAutoGluonCleaner

# Make cleaner
cleaner: ConformalAutoGluonCleaner = ConformalAutoGluonCleaner(confidence_level= 0.999, seed = 42)


# Do
fit_cleaner = cleaner.fit(df_clean)

cleaned_data: tuple[pd.DataFrame, pd.DataFrame] = fit_cleaner.transform(df_error)

/home/chandlernick/BHT/Research/conformal-data-cleaning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-10 10:05:00,950 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 5
2025-12-10 10:05:00,950 - INFO - conformal_data_cleaning.cleaner.autogluon: Start fitting predictor #1 of 5
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          8
Memory Avail:       4.59 GB / 11.46 GB (40.0%)
Disk Space Avail:   155.50 GB / 467.89 GB (33.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the 

[('preprocess', ColumnTransformer(sparse_threshold=0,
                  transformers=[('categorical_features',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['feature_c', 'feature_e']),
                                ('scaled_numeric', StandardScaler(),
                                 ['feature_a', 'feature_b', 'feature_d'])])), ('predictor', <autogluon.tabular.predictor.predictor.TabularPredictor object at 0x700920eac6b0>)]


Fitting model: RandomForestMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.1342	 = Validation score   (-pinball_loss)
	0.36s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=4, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.4.0`.
Fitting model: ExtraTreesMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.1403	 = Validation score   (-pinball_loss)
	0.32s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Failed to import

[('preprocess', ColumnTransformer(sparse_threshold=0,
                  transformers=[('categorical_features',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['feature_c', 'feature_e']),
                                ('scaled_numeric', StandardScaler(),
                                 ['feature_a', 'feature_b', 'feature_d'])])), ('predictor', <autogluon.tabular.predictor.predictor.TabularPredictor object at 0x70092242dbb0>)]


Fitting model: RandomForestMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.4489	 = Validation score   (-pinball_loss)
	0.32s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=4, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.4.0`.
Fitting model: ExtraTreesMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.3906	 = Validation score   (-pinball_loss)
	0.32s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Failed to import

[('preprocess', ColumnTransformer(sparse_threshold=0,
                  transformers=[('categorical_features',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['feature_c', 'feature_e']),
                                ('scaled_numeric', StandardScaler(),
                                 ['feature_a', 'feature_b', 'feature_d'])])), ('predictor', <autogluon.tabular.predictor.predictor.TabularPredictor object at 0x700920bc9670>)]


Fitting model: RandomForestMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.3292	 = Validation score   (-pinball_loss)
	0.32s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=4, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.4.0`.
Fitting model: ExtraTreesMSE ...
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting with cpus=8, gpus=0
	-0.3344	 = Validation score   (-pinball_loss)
	0.3s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Failed to import 

In [4]:
data, mask = cleaned_data

In [5]:
(data != df_error).sum()

feature_a    17
feature_b    13
feature_c    20
feature_d     9
feature_e     0
dtype: int64

In [6]:
mask.sum()

feature_a    17
feature_b    13
feature_c    20
feature_d     9
feature_e     0
dtype: int64